## MMTHE01 - Masters Thesis

### F3C. XAI Sample Based Evaluation with the ANN Model (Propagation based XAI techniques)

* * Sample based evaluation of XAI techniques on the ANN Model: Integrated Gradient

#### Importing the libraries

In [1]:
### import general libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import time
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import re

from sklearn.preprocessing import LabelEncoder

In [2]:
### Import libraries to save models
import pickle
from tensorflow.keras.models import Model, load_model
from scipy.stats import pearsonr
import warnings

In [3]:
## Check your current working directory
cwd = os.getcwd()

# Build the full path
full_path = r"C:\Users\eaber\Documents\Master Thesis Final\Analysis"

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

Folder does not exist: C:\Users\eaber\Documents\Master Thesis Final\Analysis


#### Load Dataset

In [4]:
with open("saved_data/Oversampled_datasets_folder.pkl","rb") as f:
    X_train_np, y_train_np, X_test_np, y_test_np, X_train_scaled_np, X_test_scaled_np = pickle.load(f)

In [5]:
### Import the pickle saved dataset
with open("saved_data/selected_features_final.pkl","rb") as f:
    selected_features_final = pickle.load(f)

In [6]:
if 'default_status' in selected_features_final:
    selected_features_final.remove('default_status')
    
feature_names = [re.sub(r"[ <>+]", "_", c).replace(" ", "_") for c in selected_features_final]

In [7]:
# Convert to dataframe
#X_train = pd.DataFrame(X_train_np, columns=feature_names).apply(pd.to_numeric)
#X_test = pd.DataFrame(X_test_np, columns=feature_names).apply(pd.to_numeric)
X_train_scaled_full = pd.DataFrame(X_train_scaled_np, columns=feature_names).apply(pd.to_numeric)
X_test_scaled_full = pd.DataFrame(X_test_scaled_np, columns=feature_names).apply(pd.to_numeric)

y_train_full = pd.DataFrame(y_train_np, columns=['default_status'])
y_test_full = pd.DataFrame(y_test_np, columns=['default_status'])

#### Obtaining samples for the evaluation (top X train, top y test)

In [8]:
X_train_scaled = X_train_scaled_full #.head(200)
X_test_scaled = X_test_scaled_full #.head(40)

In [9]:
y_train = y_train_full #.head(200)
y_test = y_test_full #.head(40)

#### Loading the models

In [10]:
# load ANN model
model = load_model("saved_models/ann_model_Oversampled.keras")

In [11]:
#X_test_batch = X_test_scaled.values[:2048]  # for testing, full run will use all values
#y_test_batch = y_test.values[:2048] 

In [12]:
import tensorflow as tf

In [13]:
## Define index for local tests
idx = 1

### 6 Sample based equation of XAI techniques for the ANN Model

#### 6.0 Define all evaluation functions for Local Integrated Gradient

In [14]:
def resolve_feature_name(feature_name, X_columns):
    """
    Maps XAI feature names (SHAP / LIME) to raw feature names in X.
    """

    # Exact match (SHAP or already-clean LIME)
    if feature_name in X_columns:
        return feature_name

    # LIME numeric rule: "feature <= value", "feature > value"
    base = re.split(r"\s*(<=|>=|<|>|=)\s*", feature_name)[0]

    if base in X_columns:
        return base

    # If still not found, return None (skip feature)
    return None

In [15]:
# Incompleteness Metric
def faithfulness_metric(
    model,
    X,
    feature_importances_df,
    background_df,
    value='importance',
    abs_value='abs_importance',
    correlation_type='pearson',
    baseline='mean',
    min_variance=1e-6,
    step_size=1
):
    """
    Compute the faithfulness of a local feature-attribution explanation.

    Faithfulness is measured as the correlation between feature importance
    scores and the corresponding change in model prediction when each
    feature is perturbed independently.

    This implementation is model-agnostic and supports explanation methods
    that use either raw feature names (e.g., SHAP) or rule-based feature
    representations (e.g., LIME). For rule-based explanations, perturbations
    are applied to the underlying input feature rather than the rule boundary.

    Parameters
    ----------
    model : object
        Trained model with a `predict()` method.
    X : pandas.DataFrame
        Single-row input instance for which faithfulness is evaluated.
    feature_importances_df : pandas.DataFrame
        Local explanation output containing feature names and importance
        scores (e.g., from SHAP or LIME).
    background_df : pandas.DataFrame
        Reference dataset used to compute baseline feature values.
    value : str, default='importance'
        Column name containing signed feature importance values.
    abs_value : str, default='abs_importance'
        Column name containing absolute feature importance values.
    correlation_type : 'pearson'
        Correlation metric used to quantify faithfulness.
    baseline : {'mean', 'median'}, default='mean'
        Statistic used to compute baseline feature values.
    min_variance : float, default=1e-6
        Minimum feature variance required to apply perturbation.
    step_size : float, default=1.0
        Scaling factor applied when perturbing features toward the baseline.

    Returns
    -------
    float
        Faithfulness correlation score. Returns NaN if insufficient
        valid feature perturbations are available.
    """

    # Original prediction
    #y_orig = model.predict(X)[0]
    y_orig = float(np.squeeze(model.predict(X.values)))

    # Baseline values
    if baseline == 'mean':
        baseline_values = background_df.mean()
    elif baseline == 'median':
        baseline_values = background_df.median()
    else:
        raise ValueError("baseline must be 'mean' or 'median'")

    predicted_changes = []
    importances = []

    x_orig = X.iloc[0]

    for _, row in feature_importances_df.iterrows():

        raw_feature = resolve_feature_name(row['feature'], X.columns)

        # Skip non-resolvable features
        if raw_feature is None:
            continue

        # Skip low-variance features
        if background_df[raw_feature].var() < min_variance:
            continue

        x_perturbed = x_orig.copy()

        # Perturbation
        x_perturbed[raw_feature] = baseline_values[raw_feature] * step_size

        #y_new = model.predict(pd.DataFrame([x_perturbed]))[0]
        y_new = float(np.squeeze(model.predict(pd.DataFrame([x_perturbed]).values)))

        predicted_changes.append(abs(y_orig - y_new))
        importances.append(abs(row[abs_value]))

    # Not enough signal
    if len(predicted_changes) < 2:
        return np.nan

    # Correlation
    if correlation_type == 'pearson':
        return pearsonr(predicted_changes, importances)[0]
    elif correlation_type == 'pearson':
        return pearsonr(predicted_changes, importances)[0]
    else:
        raise ValueError("correlation_type must be 'pearson'")

In [16]:
def monotonicity_metric(model, X, feature_importances_df, background_df, baseline='mean', value_col='importance', min_variance=1e-6):
    """
    Compute monotonicity of feature attributions for a single instance.

    Parameters
    ----------
    model : object
        Trained model with predict() method.
    X : pandas.DataFrame
        Single-row instance (1 x n_features).
    feature_importances_df : pandas.DataFrame
        DataFrame with columns ["feature", value_col] for the instance.
    background_df : pandas.DataFrame
        Reference dataset for computing baseline values.
    baseline : {'mean', 'median'}, default='mean'
        Statistic used to compute baseline feature values.
    value_col : str
        Column name with attribution values.
    min_variance : float
        Minimum variance threshold to skip nearly-constant features.

    Returns
    -------
    float
        Pearson correlation between |attribution| and incremental contribution.
    """

    # Ensure X is single-row DataFrame
    if isinstance(X, pd.Series):
        X = X.to_frame().T

    # Sort features by increasing attribution magnitude
    features = feature_importances_df['feature'].tolist()
    attributions = feature_importances_df[value_col].values
    sorted_idx = np.argsort(np.abs(attributions))
    sorted_features = [features[i] for i in sorted_idx]
    sorted_attributions = attributions[sorted_idx]

    # Determine baseline
    if baseline == 'mean':
        baseline_values = background_df.mean()
    elif baseline == 'median':
        baseline_values = background_df.median()
    else:
        raise ValueError("baseline must be 'mean' or 'median'")

    # Compute cumulative contributions
    cumulative_contributions = []
    x_current = baseline_values.copy()

    for f in sorted_features:
        # Resolve feature name
        raw_feature = resolve_feature_name(f, X.columns)
        if raw_feature is None:
            continue

        # Skip low-variance features
        if background_df[raw_feature].var() < min_variance:
            continue

        # Perturb feature toward actual value
        x_current[raw_feature] = X[raw_feature].values[0]
        #pred = model.predict(pd.DataFrame([x_current]))[0]
        pred = model.predict(pd.DataFrame([x_current]))[0][0]
        cumulative_contributions.append(pred)

    if not cumulative_contributions:
        return np.nan

    # Incremental contributions
    increments = np.diff([baseline_values.sum()] + cumulative_contributions)

    # Monotonicity = Pearson correlation
    corr, _ = pearsonr(np.abs(sorted_attributions[:len(increments)]), increments)
    return corr

In [17]:
def incompleteness_metric(
    model,
    X,
    df,
    background_df=None,
    base_value=None,
    value_col='importance',
    baseline='mean'
):
    """
    Compute incompleteness of a local explanation.

    Incompleteness measures the deviation between the model prediction
    and the sum of feature attributions plus a baseline prediction.

    For SHAP, the baseline corresponds to the expected value.
    For LIME, the baseline is computed as the model prediction at a
    reference input (mean or median of the background data).

    Parameters
    ----------
    model : object
        Trained model with predict().
    X : pandas.DataFrame
        Single-row input instance.
    df : pandas.DataFrame
        Local explanation with feature importance values.
    background_df : pandas.DataFrame, optional
        Reference dataset used to compute the baseline input.
        Required if base_value is not provided.
    base_value : float, optional
        Baseline prediction (e.g., SHAP expected value).
    value_col : str, default='importance'
        Column containing signed feature attributions.
    baseline : {'mean', 'median'}, default='mean'
        Statistic used to compute reference input for LIME.

    Returns
    -------
    float
        Absolute incompleteness score.
    """

    # Model prediction
    y_pred = model.predict(X)[0]

    # Determine baseline prediction
    if base_value is not None:
        baseline_pred = base_value
    else:
        if background_df is None:
            raise ValueError("background_df must be provided when base_value is None")

        if baseline == 'mean':
            x_ref = background_df.mean()
        elif baseline == 'median':
            x_ref = background_df.median()
        else:
            raise ValueError("baseline must be 'mean' or 'median'")

        baseline_pred = model.predict(pd.DataFrame([x_ref]))[0]

    # Sum of attributions
    attribution_sum = df[value_col].sum()

    # Incompleteness
    return abs(y_pred - (baseline_pred + attribution_sum))

In [18]:
# -----------------------------
# Helper: suppress all output
# -----------------------------
import io
import contextlib
import os
import sys

@contextlib.contextmanager
def suppress_output():
    """Completely suppress stdout and stderr."""
    with open(os.devnull, 'w') as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout, sys.stderr = devnull, devnull
            yield
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr

In [19]:
def explanation_to_vector(explanation, feature_names):
    """
    Convert SHAP or LIME explanation to numeric vector aligned with X.columns
    """
    vec = pd.Series(0.0, index=feature_names)

    # list
    if isinstance(explanation, list):
        for item in explanation:
            if not isinstance(item, (tuple, list)) or len(item) != 2:
                continue
            feat, weight = item
            if feat is None or str(feat).lower() in {"intercept", "bias"}:
                continue
            base_feat = str(feat).split()[0]
            if base_feat in vec.index:
                vec[base_feat] += weight

    # numpy array, list, or Series
    else:
        # Handle multi-class SHAP returning list of arrays
        if isinstance(explanation, list):
            # Take first array if list of arrays
            explanation = explanation[0]
        exp_arr = np.asarray(explanation).ravel()
        if len(exp_arr) != len(vec):
            raise ValueError(f"Length mismatch: {len(exp_arr)} vs {len(vec)}")
        vec[:] = exp_arr

    return vec.values

In [20]:
# -----------------------------
# Main: Explanation Sensitivity (SENS_MAX)
# -----------------------------
def explanation_sensitivity(
    model,
    X,
    explainer_func,
    r=1e-3,
    n_samples=100
):
    """
    Compute SENS_MAX: maximum local explanation sensitivity
    (Yeh et al., 2019).

    Measures the largest change in the explanation vector when
    the input is perturbed within a small radius r.
    Supports SHAP and LIME explanations.
    """

    x_orig = X.iloc[0]

    # Original explanation (silent)
    with suppress_output():
        feat_orig_raw = explainer_func(model, pd.DataFrame([x_orig]))

    feat_orig = explanation_to_vector(feat_orig_raw, X.columns)

    sensitivities = []

    for _ in range(n_samples):
        # Generate perturbation within radius r
        delta = np.random.normal(0, 1, size=len(x_orig))
        delta = delta / np.linalg.norm(delta) * r  # scale to have L2 norm = r

        x_perturbed = pd.DataFrame([x_orig + delta])

        # Perturbed explanation (silent)
        with suppress_output():
            feat_perturbed_raw = explainer_func(model, x_perturbed)

        feat_perturbed = explanation_to_vector(feat_perturbed_raw, X.columns)

        # L2 norm of explanation change
        diff = np.linalg.norm(feat_orig - feat_perturbed, ord=2)
        sensitivities.append(diff)

    # SENS_MAX: maximum observed sensitivity
    return np.max(sensitivities) if sensitivities else np.nan

#### 6.1 Define all evaluation functions for Global Integrated Gradient

In [21]:
def faithfulness_metric_global(model, X, feature_importances_df, base_value=None, background_df=None, baseline='mean', correlation_type='pearson'):
    """
    Compute the true global faithfulness metric for any feature-importance-based XAI method.

    Faithfulness measures whether features with higher importance actually
    cause larger changes in the model output when perturbed.

    Parameters
    ----------
    model : trained model with .predict() method
    X : pd.DataFrame
        Feature matrix (n_samples x n_features)
    feature_importances_df : pd.DataFrame
        Per-sample feature importances (n_samples x n_features)
    baseline : str, float, or pd.Series
        Value to use for perturbation. Default 'mean' replaces feature with column mean.
        Can also pass a scalar or Series with length = n_features.
    correlation_type : str
        'pearson'

    Returns
    -------
    correlations : dict
        Per-feature correlation between importance and output change
    mean_corr : float
        Mean correlation across all features
    """

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=feature_importances_df.columns)
    
    X_perturbed = X.copy()
    correlations = {}

    # Baseline values
    if base_value != None:
        baseline_values = base_value
    else:
        if baseline == 'mean':
            baseline_values = background_df.mean()
        elif baseline == 'median':
            baseline_values = background_df.median()
        else:
            raise ValueError("baseline must be 'mean' or 'median'")

    # Original predictions
    #y_orig = model.predict(X)
    #y_orig = float(np.squeeze(model.predict(X.values)))
    y_orig = model.predict(X).reshape(-1)

    for feature in feature_importances_df.columns:
        # Skip constant features
        if X[feature].nunique() < 2 or feature_importances_df[feature].nunique() < 2:
            correlations[feature] = np.nan
            continue

        # Perturb the feature to baseline
        X_perturbed[feature] = baseline_values[feature]
        
        # Model output after perturbation
        #y_perturbed = model.predict(X_perturbed)
        #y_perturbed = float(np.squeeze(model.predict(x_perturbed.values)))
        y_perturbed = model.predict(X_perturbed).reshape(-1)
        
        # Absolute change in output
        delta_y = np.abs(y_orig - y_perturbed)

        # Skip if delta_y is constant (correlation undefined)
        if np.all(delta_y == delta_y[0]):
            correlations[feature] = np.nan
            X_perturbed[feature] = X[feature]  # reset
            continue

        # Compute correlation safely
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            if correlation_type.lower() == 'pearson':
                corr, _ = pearsonr(feature_importances_df[feature], delta_y)
            elif correlation_type.lower() == 'pearson':
                corr, _ = pearsonr(feature_importances_df[feature], delta_y)
            else:
                raise ValueError("correlation_type must be 'pearson'")

        correlations[feature] = corr
        X_perturbed[feature] = X[feature]  # reset for next feature

    mean_corr = np.nanmean(list(correlations.values()))
    return mean_corr

In [22]:
# Incompleteness Metric
def incompleteness_metric_global(
    model,
    X,
    feature_importances_df,
    background_df=None,
    base_value=None,
    value_col=None,
    baseline='mean'
):
    """
    Compute incompleteness of a global explanation.

    Incompleteness measures the deviation between the model prediction
    and the sum of feature attributions plus a baseline prediction.

    Parameters
    ----------
    model : object
        Trained model with .predict().
    X : pd.DataFrame
        Input feature matrix (n_samples x n_features).
    feature_importances_df : pd.DataFrame
        Per-sample feature importances (n_samples x n_features).
        Columns must match X.columns. If `value_col` is provided, uses that column instead.
    background_df : pd.DataFrame, optional
        Reference dataset used to compute baseline input if `base_value` is None.
    base_value : float, optional
        Baseline prediction (e.g., SHAP expected value). If provided, used for all samples.
    value_col : str, optional
        If feature_importances_df is in long format, the column containing attribution values.
    baseline : {'mean', 'median'}, default='mean'
        Statistic used to compute reference input for baseline prediction.

    Returns
    -------
    incompleteness_per_sample : np.ndarray
        Absolute incompleteness score per sample.
    mean_incompleteness : float
        Mean incompleteness across all samples.
    """

    # Original predictions for all samples
    y_pred = model.predict(X)

    # Determine baseline prediction
    if base_value is not None:
        #baseline_pred = base_value
        baseline_pred = float(model.predict(baseline_df).reshape(-1)[0])
    else:
        if background_df is None:
            raise ValueError("background_df must be provided when base_value is None")
        if baseline == 'mean':
            x_ref = background_df.mean()
        elif baseline == 'median':
            x_ref = background_df.median()
        else:
            raise ValueError("baseline must be 'mean' or 'median'")
        baseline_pred = model.predict(pd.DataFrame([x_ref]))[0]

    # Sum of attributions per sample
    if value_col is not None:
        # long-format DataFrame with feature/value columns
        attribution_sum = feature_importances_df[value_col].groupby(feature_importances_df.index).sum().values
    else:
        # wide-format DataFrame (n_samples x n_features)
        attribution_sum = feature_importances_df.sum(axis=1).values

    # Incompleteness per sample
    incompleteness_per_sample = np.abs(y_pred - (baseline_pred + attribution_sum))

    # Mean incompleteness across samples
    mean_incompleteness = np.mean(incompleteness_per_sample)

    return mean_incompleteness

#### 6.7 Evaluating Integrated Gradients (IG) to the ANN Model
* F3E. Thesis - Evaludate different XAI techniques with the ANN Model (IG)

In [23]:
# Integrated Gradients function
@tf.function
def interpolate_inputs(baseline, input, alphas):
    alphas_x = alphas[:, tf.newaxis] * (input - baseline) + baseline
    return alphas_x

def integrated_gradients(model, input, X_train=None, steps=50):
    if X_train is None:
        # fallback to zeros if training data not provided
        baseline = tf.zeros_like(input)  # all zeros baseline
    else:
        baseline = tf.convert_to_tensor([X_train.mean().values], dtype=tf.float32)
    
    # Generate alphas from 0 to 1
    alphas = tf.linspace(0.0, 1.0, steps)
    
    # Interpolated inputs
    interpolated_inputs = interpolate_inputs(baseline, input, alphas)
    
    # Gradient function
    with tf.GradientTape() as tape:
        tape.watch(interpolated_inputs)
        preds = model(interpolated_inputs)
    grads = tape.gradient(preds, interpolated_inputs)
    
    # Average gradients and scale
    avg_grads = tf.reduce_mean(grads, axis=0)
    integrated_grad = (input - baseline) * avg_grads
    return integrated_grad.numpy()

In [24]:
baseline_df = pd.DataFrame([X_train_scaled.mean(axis=0)], columns=X_train_scaled.columns)

##### Local implementation of IG

In [25]:
# Get the first row as a 2D array (batch of 1)
idx=1
x_sample = tf.convert_to_tensor(X_test_scaled.iloc[[idx]], dtype=tf.float32)

print(x_sample.shape)  # (1, n_features)

(1, 109)


In [26]:
local_ig = integrated_gradients(model, x_sample, X_train=baseline_df, steps=50)[0]

In [27]:
local_df = pd.DataFrame({
    "feature": X_test_scaled.columns,
    "attribution": local_ig,
    "abs_attribution": abs(local_ig)
}).sort_values(by="attribution", key=np.abs, ascending=False)

##### Evaluation of local integrated gradient

In [28]:
# Evaluate Faithfulness
np.random.seed(1)
faithfulness_score = faithfulness_metric(
    model,
    X_test_scaled.iloc[[idx]],
    local_df,
    background_df=baseline_df,
    value='attribution',
    abs_value='abs_attribution',
    correlation_type='pearson'
)
print("Faithfulness correlation score Integrated Gradient Local:", faithfulness_score)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━

In [29]:
np.random.seed(1)
monotonicity = monotonicity_metric(
    model=model,
    X=X_test_scaled.iloc[[idx]],
    feature_importances_df=local_df,
    background_df=baseline_df,
    value_col='attribution')

print("Monotonicity SHAP Local:", monotonicity)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━

In [30]:
np.random.seed(1)
incompleteness = incompleteness_metric(
    model=model,
    X=X_test_scaled.iloc[[idx]],
    df=local_df,
    background_df=baseline_df,
    value_col='attribution'
)

print("Incompleteness score Integrated Gradient Local:", incompleteness)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Incompleteness score Integrated Gradient Local: [0.00261928]


In [31]:
# Define explainer_func correctly: only input X
np.random.seed(1)
def ig_explainer_func(
    model,
    X,
    X_train=baseline_df,
    steps=50
):
    """
    Integrated Gradients explainer function.

    Parameters
    ----------
    model : tf.keras.Model
    X : pd.DataFrame
        One or more samples
    baseline : tf.Tensor or None
    steps : int

    Returns
    -------
    np.ndarray
        Shape = (n_samples, n_features)
        Raw IG attributions (signed)
    """

    X_tensor = tf.convert_to_tensor(
        X.values,
        dtype=tf.float32
    )

    if X_train is None:
        # fallback to zeros if training data not provided
        baseline = tf.zeros_like(input)  # all zeros baseline
    else:
        baseline = tf.convert_to_tensor([X_train.mean().values], dtype=tf.float32)

    ig_all = []

    for i in range(X_tensor.shape[0]):
        ig_vals = integrated_gradients(
            model,
            X_tensor[i:i+1],
            X_train=baseline_df,
            steps=steps
        )[0]  # remove batch dim

        ig_all.append(ig_vals)

    return np.vstack(ig_all)

In [32]:
np.random.seed(1)
sensitivity_score  = explanation_sensitivity(
    model,
    X_test_scaled.iloc[[idx]],
    ig_explainer_func,
    r=1e-3,
    n_samples=30)

print("Explanation Sensitivity score Integrated Gradient Local:", sensitivity_score)

Explanation Sensitivity score Integrated Gradient Local: 3.98952581698966e-05


##### Global implementation of IG

In [33]:
len(X_test_scaled)

45214

In [34]:
ig_attributions = []
n_samples = len(X_test_scaled)

In [35]:
for i in range(n_samples):
    x = tf.convert_to_tensor([X_test_scaled.iloc[i].values], dtype=tf.float32)
    ig_attr = integrated_gradients(model, x, X_train=baseline_df, steps=50)
    ig_attributions.append(ig_attr[0])

In [36]:
ig_attributions = np.array(ig_attributions)

In [37]:
# Convert to DataFrame (n_samples x n_features)
global_df = pd.DataFrame(ig_attributions, columns=X_test_scaled.columns)

##### Evaluation of Global IG

In [38]:
global_df.shape

(45214, 109)

In [39]:
X_test_scaled.shape

(45214, 109)

In [40]:
# Evaluate Faithfulness
np.random.seed(1)
faithfulness_score = faithfulness_metric_global(
    model=model,
    X=X_test_scaled,
    feature_importances_df=global_df,
    background_df=baseline_df,
    correlation_type='pearson'
)

print("Faithfulness correlation score Global Integrated Gradient:", faithfulness_score)

1413/1413 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1413/1413 ━

In [41]:
# Monotonicity metric
# Only run for local methods as it requires extensive resources to run over the full 45,214 test datasets to obtain a global metric.

In [42]:
np.random.seed(1)
incompleteness = incompleteness_metric_global(
    model=model,
    X=X_test_scaled,
    feature_importances_df=global_df,
    base_value=baseline_df,
)

print("Incompleteness score Global Integrated Gradient:", incompleteness)

1413/1413 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Incompleteness score Global Integrated Gradient: 0.17269875


In [43]:
# Sensitivity metric
# Only run for local methods as it requires extensive resources to run over the full 45,214 test datasets to obtain a global metric.